In [23]:
import os
import open3d as o3d
import pdal
import numpy as np
import json
import laspy
from config import *

In [24]:
data_dir = input_laz_dir


In [25]:
gujarath_data = [os.path.join(data_dir,i) for i in os.listdir(data_dir) if i.endswith('.las')]
gujarath_data

['/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las']

In [26]:
pdal_pipeline = {
    "pipeline": [
        {
            "type": "readers.las",
            "filename": str(gujarath_data[0])  # explicitly cast to str, e.g. if it's a Path object
        },
        {
            "type": "filters.smrf"
        },
        {
            "type": "filters.range",
            "limits": "Classification[2:2]"
        }
    ]
}

In [27]:
pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
pipeline.execute()
arrays = pipeline.arrays
ground_points = arrays[0]

print(f"Ground point count: {len(ground_points)}")
print(f"Fields available: {ground_points.dtype.names}")

x = ground_points["X"]
y = ground_points["Y"]
z = ground_points["Z"]


Ground point count: 42675461
Fields available: ('X', 'Y', 'Z', 'Intensity', 'ReturnNumber', 'NumberOfReturns', 'ScanDirectionFlag', 'EdgeOfFlightLine', 'Classification', 'Synthetic', 'KeyPoint', 'Withheld', 'Overlap', 'ScanAngleRank', 'UserData', 'PointSourceId', 'Red', 'Green', 'Blue')


In [28]:
header = laspy.LasHeader(point_format=1, version="1.2")
las = laspy.LasData(header=header)

las.x = ground_points["X"]
las.y = ground_points["Y"]
las.z = ground_points["Z"]

output_las = os.path.join(output_dir,"ground_points.las")
las.write(output_las)

In [29]:
points = np.column_stack((las.x, las.y, las.z))
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

In [30]:
o3d.visualization.draw_geometries([pcd])